# Boundary Crossing Audit Why 2,252 Flips Don't Change Behavior

The Neagari search found 2,252 beneficial flips with 27-47% acceptance rates.
But **how many of those flips actually changed what the model says?**

Each flip has per-probe logit gap snapshots (before/after). A "boundary crossing"
is when a probe's gap goes from negative (model says the wrong token) to positive
(model says the right token), or vice versa. Only boundary crossings change behavior.

**No GPU needed.** All data is in the patch JSON files.

In [ ]:
# Setup, clone repo if needed
import os
if not os.path.exists('neagari'):
    !git clone https://github.com/sbenjam1n/neagari.git
os.chdir('neagari')
!pip install -q matplotlib numpy 2>/dev/null

In [ ]:
import json, glob, os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Load all patch files
all_domains = []
for d in ['v2_patches', 'v3_patches']:
    for pf in sorted(glob.glob(os.path.join(d, 'patch_*.json'))):
        data = json.load(open(pf))
        domain = data.get('domain', '?')
        source = os.path.basename(d)
        flips = data.get('flips', [])
        if flips and 'target_gaps_before' in flips[0]:
            all_domains.append({
                'key': f'{source}/{domain}',
                'source': source,
                'domain': domain,
                'flips': flips
            })

total_flips = sum(len(d['flips']) for d in all_domains)
print(f'Loaded {total_flips} flips across {len(all_domains)} domain runs')

## 1. The Headline: How many flips cross the boundary?

In [ ]:
# Count boundary crossings across all flips
total = 0
crossings = 0
fixes = 0
breaks = 0

for dom in all_domains:
    for flip in dom['flips']:
        total += 1
        t_before = {n: g for n, g in flip['target_gaps_before']}
        t_after = {n: g for n, g in flip['target_gaps_after']}
        has_cross = False
        for name in t_before:
            b, a = t_before[name], t_after.get(name, t_before[name])
            if b <= 0 and a > 0:
                fixes += 1
                has_cross = True
            elif b > 0 and a <= 0:
                breaks += 1
                has_cross = True
        if has_cross:
            crossings += 1

print(f'Total accepted flips: {total:,}')
print(f'Flips with any boundary crossing: {crossings} ({100*crossings/total:.2f}%)')
print(f'Total fix events (wrong→right): {fixes}')
print(f'Total break events (right→wrong): {breaks}')
print(f'\nFlips that changed NOTHING about model behavior: {total - crossings:,} ({100*(total-crossings)/total:.1f}%)')

In [ ]:
# Visual: the ratio
fig, ax = plt.subplots(1, 1, figsize=(10, 2))

ax.barh([0], [total - crossings], color='#d4d4d4', height=0.6, label=f'No crossing ({total-crossings:,})')
ax.barh([0], [crossings], left=[total - crossings], color='#e74c3c', height=0.6, label=f'Boundary crossing ({crossings})')

ax.set_xlim(0, total)
ax.set_yticks([])
ax.set_xlabel('Accepted flips')
ax.set_title(f'Of {total:,} accepted flips, {crossings} changed model behavior', fontsize=14, fontweight='bold')
ax.legend(loc='upper right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)
plt.tight_layout()
plt.show()

## 2. Why: the gap between per-flip deltas and decision margins

In [ ]:
# Collect all per-flip per-probe deltas and all baseline gap magnitudes
all_deltas = []
all_baseline_gaps = []
domain_stats = []

for dom in all_domains:
    deltas = []
    flips = dom['flips']

    # Baseline gaps from first flip's 'before'
    baseline = {n: g for n, g in flips[0]['target_gaps_before']}
    for g in baseline.values():
        all_baseline_gaps.append(g)

    for flip in flips:
        t_b = {n: g for n, g in flip['target_gaps_before']}
        t_a = {n: g for n, g in flip['target_gaps_after']}
        for name in t_b:
            d = t_a.get(name, t_b[name]) - t_b[name]
            deltas.append(d)
            all_deltas.append(d)

    deltas = np.array(deltas)
    bl = np.array(list(baseline.values()))
    wrong_gaps = bl[bl <= 0]
    closest_wrong = np.min(np.abs(wrong_gaps)) if len(wrong_gaps) > 0 else None

    domain_stats.append({
        'key': dom['key'],
        'flips': len(flips),
        'probes': len(baseline),
        'mean_delta': deltas.mean(),
        'closest_wrong': closest_wrong,
        'flips_needed': int(closest_wrong / max(deltas.mean(), 1e-10)) if closest_wrong else None,
    })

print(f'{"Domain":<30} {"Flips":>6} {"Mean Δ":>12} {"Closest wrong":>14} {"Flips to cross":>15}')
print('─' * 80)
for s in domain_stats:
    fn = f'{s["flips_needed"]:,}' if s['flips_needed'] else 'N/A'
    cw = f'{s["closest_wrong"]:.4f}' if s['closest_wrong'] else 'N/A'
    print(f'{s["key"]:<30} {s["flips"]:>6} {s["mean_delta"]:>12.6f} {cw:>14} {fn:>15}')

In [ ]:
# Histogram: per-flip deltas vs decision margins
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: distribution of per-flip deltas
d = np.array(all_deltas)
# Clip for visibility
d_clip = np.clip(d, -0.01, 0.01)
ax1.hist(d_clip, bins=100, color='steelblue', alpha=0.8, edgecolor='none')
ax1.axvline(x=0, color='red', linestyle='--', alpha=0.5)
ax1.set_xlabel('Per-flip per-probe logit delta')
ax1.set_ylabel('Count')
ax1.set_title(f'Per-flip deltas (clipped to ±0.01)\nMean = {d.mean():.6f}', fontsize=12)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Right: distribution of baseline gaps (how far from boundary)
bl = np.array(all_baseline_gaps)
wrong = bl[bl <= 0]
right = bl[bl > 0]
ax2.hist(wrong, bins=40, color='#e74c3c', alpha=0.7, label=f'Wrong ({len(wrong)})', edgecolor='none')
ax2.hist(right, bins=40, color='#27ae60', alpha=0.7, label=f'Right ({len(right)})', edgecolor='none')
ax2.axvline(x=0, color='black', linestyle='-', linewidth=2)
ax2.set_xlabel('Baseline logit gap')
ax2.set_ylabel('Count')
ax2.set_title(f'Baseline gaps (boundary at 0)\nProbes near boundary (|gap|<1): {np.sum(np.abs(bl)<1)}', fontsize=12)
ax2.legend()
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print(f'\nThe average flip moves a probe by {abs(d.mean()):.6f} logit points.')
print(f'The average wrong probe is {abs(wrong.mean()):.2f} logit points from the boundary.')
print(f'Ratio: {abs(wrong.mean()) / max(abs(d.mean()), 1e-10):.0f}x — flips are ~{abs(wrong.mean()) / max(abs(d.mean()), 1e-10):.0f}x too weak to cross.')

## 3. Probe-level trajectory: watching gaps evolve across 2,252 flips

In [ ]:
# Track gap trajectories for the 5 closest-to-boundary wrong probes in each domain
fig, axes = plt.subplots(2, 5, figsize=(20, 8), sharey=False)
axes = axes.flatten()

plot_idx = 0
for dom in all_domains:
    if plot_idx >= 10:
        break
    flips = dom['flips']

    # Get all probe names and their baseline gaps
    baseline = {n: g for n, g in flips[0]['target_gaps_before']}
    # Sort by distance to boundary
    wrong_probes = sorted(
        [(n, g) for n, g in baseline.items() if g <= 0],
        key=lambda x: abs(x[1])
    )[:3]  # top 3 closest to boundary

    if not wrong_probes:
        plot_idx += 1
        continue

    ax = axes[plot_idx]

    for probe_name, _ in wrong_probes:
        trajectory = []
        for flip in flips:
            gaps_after = {n: g for n, g in flip['target_gaps_after']}
            trajectory.append(gaps_after.get(probe_name, 0))

        short_name = probe_name.split('_')[-1][:10] if len(probe_name) > 15 else probe_name
        ax.plot(trajectory, linewidth=0.8, alpha=0.8, label=short_name)

    ax.axhline(y=0, color='red', linestyle='--', alpha=0.5, linewidth=1)
    ax.set_title(dom['key'].replace('_patches/', '\n'), fontsize=9)
    ax.set_xlabel('Flip #', fontsize=8)
    if plot_idx % 5 == 0:
        ax.set_ylabel('Logit gap', fontsize=8)
    ax.tick_params(labelsize=7)
    ax.legend(fontsize=6, loc='best')

    plot_idx += 1

# Hide unused axes
for i in range(plot_idx, 10):
    axes[i].set_visible(False)

plt.suptitle('Closest-to-boundary probes: gap trajectories across search\n(red line = decision boundary)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print('Each line is a probe\'s gap evolving as flips accumulate.')
print('Lines that never touch the red line = no behavior change.')

## 4. Per-domain breakdown

In [ ]:
# Detailed per-domain analysis
print(f'{"Domain":<28} {"Flips":>6} {"Probes":>7} {"Wrong":>6} {"Border":>7} {"Mean Δ":>10} {"Closest":>9} {"Need":>8}')
print('═' * 90)

grand_total_flips = 0
grand_total_wrong = 0
grand_total_borderline = 0

for dom in all_domains:
    flips = dom['flips']
    baseline = {n: g for n, g in flips[0]['target_gaps_before']}
    final = {n: g for n, g in flips[-1]['target_gaps_after']}

    bl = np.array(list(baseline.values()))
    fn = np.array(list(final.values()))

    wrong = np.sum(bl <= 0)
    borderline = np.sum((bl > -2.0) & (bl <= 0))

    deltas = []
    for flip in flips:
        tb = {n: g for n, g in flip['target_gaps_before']}
        ta = {n: g for n, g in flip['target_gaps_after']}
        for name in tb:
            deltas.append(ta.get(name, tb[name]) - tb[name])
    mean_d = np.mean(deltas)

    wrong_gaps = bl[bl <= 0]
    closest = np.min(np.abs(wrong_gaps)) if len(wrong_gaps) > 0 else 0
    needed = int(closest / max(abs(mean_d), 1e-10))

    grand_total_flips += len(flips)
    grand_total_wrong += wrong
    grand_total_borderline += borderline

    # How many flips between start and end did the closest probe actually move?
    closest_name = sorted([(n, abs(g)) for n, g in baseline.items() if g <= 0], key=lambda x: x[1])[0][0] if wrong > 0 else None
    if closest_name:
        actual_move = final.get(closest_name, 0) - baseline[closest_name]
    else:
        actual_move = 0

    print(f'{dom["key"]:<28} {len(flips):>6} {len(baseline):>7} {wrong:>6} '
          f'{borderline:>7} {mean_d:>10.6f} {closest:>9.4f} {needed:>8,}')

print('─' * 90)
print(f'\nInterpretation:')
print(f'  "Closest" = distance from the nearest wrong probe to the decision boundary.')
print(f'  "Need" = flips required at mean delta rate to cross the boundary for that one probe.')
print(f'  "Border" = probes within gap [-2.0, 0] the reachable zone.')
print(f'\n  The search runs 200-1000 iterations but typically needs 1,000-100,000+ to cross.')
print(f'  The fitness function accepts flips that improve the AVERAGE by 0.0001,')
print(f'  but never concentrates effort on the one probe that\'s closest to flipping.')

## 5. The fix: boundary-crossing fitness function

The current fitness rewards average gap improvement equally across all probes.
A probe at gap=-8.0 that improves by +0.001 counts the same as a probe at gap=-0.01
that improves by +0.001. But only the second one is close to actually changing behavior.

Three alternatives:

In [ ]:
# Simulate: how would different fitness functions have scored the same flips?
import textwrap

def fitness_current(tg_before, tg_after, t_baseline, cg_before, cg_after, c_baseline, lam=2.0):
    """Current: average gap improvement."""
    t_imp = sum(a - b for (_, a), (_, b) in zip(tg_after, t_baseline)) / len(tg_after)
    c_deg = sum(max(0, b - a) for (_, a), (_, b) in zip(cg_after, c_baseline)) / len(cg_after)
    return t_imp - lam * c_deg

def fitness_crossing(tg_before, tg_after, t_baseline, cg_before, cg_after, c_baseline, lam=2.0):
    """Boundary crossing: count actual argmax flips."""
    fixes = sum(1 for (_, b), (_, a) in zip(tg_before, tg_after) if b <= 0 and a > 0)
    breaks_t = sum(1 for (_, b), (_, a) in zip(tg_before, tg_after) if b > 0 and a <= 0)
    breaks_c = sum(1 for (_, b), (_, a) in zip(cg_before, cg_after) if b > 0 and a <= 0)
    return (fixes - breaks_t) - lam * breaks_c

def fitness_borderline(tg_before, tg_after, t_baseline, cg_before, cg_after, c_baseline, lam=2.0):
    """Hybrid: proximity-weighted improvement + boundary crossing bonus."""
    t_imp = sum(
        max(0, a - b) / (0.1 + abs(b))
        for (_, b), (_, a) in zip(tg_before, tg_after)
    ) / len(tg_after)
    fixes = sum(1 for (_, b), (_, a) in zip(tg_before, tg_after) if b <= 0 and a > 0)
    breaks_c = sum(1 for (_, b), (_, a) in zip(cg_before, cg_after) if b > 0 and a <= 0)
    return t_imp + 0.5 * fixes - lam * breaks_c

# Score every flip under all three fitness functions
scores = {'current': [], 'crossing': [], 'borderline': []}

for dom in all_domains:
    flips = dom['flips']
    t_bl = flips[0]['target_gaps_before']
    c_bl = flips[0]['control_gaps_before']

    for flip in flips:
        tb = flip['target_gaps_before']
        ta = flip['target_gaps_after']
        cb = flip['control_gaps_before']
        ca = flip['control_gaps_after']

        scores['current'].append(fitness_current(tb, ta, t_bl, cb, ca, c_bl) > 0)
        scores['crossing'].append(fitness_crossing(tb, ta, t_bl, cb, ca, c_bl) > 0)
        scores['borderline'].append(fitness_borderline(tb, ta, t_bl, cb, ca, c_bl) > 0)

print('How many of the 2,252 accepted flips would each fitness function accept?')
print()
for name, vals in scores.items():
    accepted = sum(vals)
    print(f'  {name:<15} {accepted:>5} / {len(vals)} ({100*accepted/len(vals):.1f}%)')

print()
print(textwrap.dedent("""
    The crossing function would accept almost nothing because almost no single
    flip crosses a boundary. But that's the point: under crossing fitness, the search
    would keep trying until it finds the rare flip that actually matters. The acceptance
    rate drops from 40% to <1%, but every accepted flip changes model behavior.

    The borderline function is the practical compromise: it gives credit for moving
    toward the boundary (so the search has gradient signal) while giving a bonus for
    actually crossing. Probes near the boundary get disproportionate weight.
""").strip())

## 6. Conclusion

The navigable degeneracy finding is **real**: 40% of random group flips improve the average logit gap.
But the average gap improvement is ~0.0001 per flip per probe, while decision boundaries are 1-10 logit points wide.

**The current search is walking laterally across a plateau.** It finds millions of improving directions,
but each step is ~10,000x too small to reach the cliff edge where behavior changes.

**The fix is in the fitness function, not the search.** Three options:
1. **Borderline probe filter** only include probes with |gap| < 2.0 in the fitness function
2. **Boundary-crossing fitness** score flips by whether they flip any probe's argmax
3. **Proximity-weighted hybrid** weight each probe's improvement by 1/(0.1 + |gap|)

Any of these would concentrate the search on the probes that can actually be fixed,
turning the structural degeneracy finding into practical behavioral improvement.